# Test Salamandra 7B per ATNE
**Objectiu**: Provar la qualitat de Salamandra (BSC) per adaptació de textos educatius en català.

**Requisits**: Executar amb GPU (Runtime > Change runtime type > T4 GPU)

---

## 1. Instal·lació i càrrega del model

In [ ]:
!pip install -q transformers accelerate torch

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import time

MODEL_ID = "BSC-LT/salamandra-7b-instruct"

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CAP'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")
print(f"\nCarregant {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
print(f"Model carregat! Parametres: {model.num_parameters()/1e9:.1f}B")

## 2. Funció de generació

In [ ]:
def generate(system_prompt, user_prompt, max_tokens=2048, temperature=0.3):
    """Genera text amb Salamandra."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
        )
    t1 = time.time()
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    words = len(response.split())
    print(f"[{t1-t0:.1f}s | {words} paraules]")
    return response

## 3. Test bàsic (verificar que funciona)

In [ ]:
resp = generate(
    "Ets un assistent que respon en català.",
    "Quin és el color del cel?",
    max_tokens=50
)
print(resp)

## 4. Test d'adaptació educativa — Primària (P1: Nouvingut àrab pre-A1)

In [ ]:
TEXT_FOTOSINTESI = """Les plantes no s'alimenten d'altres éssers vius, sinó que es fabriquen el seu propi aliment a través d'un procés anomenat fotosíntesi. Per dur a terme aquest procés, les plantes necessiten quatre elements: llum solar, aigua, diòxid de carboni i sals minerals.

El procés funciona de la manera següent. Primer, les arrels de la planta absorbeixen aigua i sals minerals de la terra. Aquesta barreja, que s'anomena saba bruta, es transporta des de les arrels fins a les fulles a través de la tija. Un cop a les fulles, la planta absorbeix el diòxid de carboni de l'aire.

A les fulles es troba un pigment de color verd anomenat clorofil·la. La clorofil·la atrapa l'energia lluminosa del sol i la transforma en energia química. Gràcies a aquesta energia, l'aigua de la saba bruta i el diòxid de carboni es converteixen en aliment per a la planta, que s'anomena saba elaborada. Durant aquest procés, la planta produeix oxigen i l'expulsa a l'aire a través de les fulles.

Finalment, la saba elaborada es distribueix per tota la planta per alimentar-la. L'oxigen que les plantes alliberen durant la fotosíntesi és fonamental per a la vida de tots els éssers vius del planeta, ja que és l'oxigen que nosaltres respirem."""

SYSTEM_P1 = """Ets un assistent expert en adaptació de textos educatius.
Adapta el text per a un alumne nouvingut àrab pre-A1 DUA Accés, primària.
Escriu en català.
Fes: prelliçó, glossari (amb traducció àrab si pots), i text adaptat.
Frases molt curtes (3-5 paraules). Una idea per frase.
Termes tècnics en negreta amb explicació al costat."""

print("=" * 60)
print("SALAMANDRA — P1: Nouvingut àrab pre-A1")
print("=" * 60)
resp_p1 = generate(SYSTEM_P1, f"Adapta el text següent:\n\n{TEXT_FOTOSINTESI}")
print(resp_p1)

## 5. Test d'adaptació educativa — P7: Altes capacitats B2 Enriquiment

In [ ]:
SYSTEM_P7 = """Ets un assistent expert en adaptació de textos educatius.
Adapta el text per a un alumne d'altes capacitats B2 DUA Enriquiment, primària.
NO simplifiquis. ENRIQUEIX: vocabulari tècnic avançat, connexions interdisciplinars,
pensament crític, preguntes de nivell alt (analitzar, avaluar, crear).
Escriu en català."""

print("=" * 60)
print("SALAMANDRA — P7: Altes capacitats B2 Enriquiment")
print("=" * 60)
resp_p7 = generate(SYSTEM_P7, f"Adapta el text següent:\n\n{TEXT_FOTOSINTESI}")
print(resp_p7)

## 6. Test d'adaptació — ESO2 Tectònica de plaques (P5: Dislèxia A2)

In [ ]:
TEXT_TECTONICA = """La tectònica de plaques és la hipòtesi segons la qual la part superficial de la Terra, anomenada litosfera, és formada per plaques rígides d'un centenar de quilòmetres de gruix que "suren" damunt l'astenosfera, una capa més plàstica i deformable. Aquestes plaques poden tenir escorça oceànica, continental, o totes dues alhora.

La litosfera es divideix en un petit nombre de grans plaques i diverses microplaques, que es comporten de manera aproximadament rígida i es mouen lentament, a velocitats de l'ordre de centímetres per any. Aquests moviments es mantenen durant milions d'anys i són de tres tipus: separació, convergència i lliscament lateral. Els límits de plaques concentren els processos geològics més importants del planeta."""

SYSTEM_P5 = """Ets un assistent expert en adaptació de textos educatius.
Adapta el text per a un alumne amb dislèxia, A2 DUA Accés, ESO.
Escriu en català. Fes glossari i text adaptat.
Frases curtes i simples. Vocabulari freqüent.
Termes tècnics en negreta amb definició al costat."""

print("=" * 60)
print("SALAMANDRA — P5: Dislèxia A2")
print("=" * 60)
resp_p5 = generate(SYSTEM_P5, f"Adapta el text següent:\n\n{TEXT_TECTONICA}")
print(resp_p5)

## 7. Resum de qualitat

In [ ]:
print("=" * 60)
print("RESUM")
print("=" * 60)
for name, resp in [("P1 Nouvingut", resp_p1), ("P7 Altes Cap.", resp_p7), ("P5 Dislèxia", resp_p5)]:
    words = len(resp.split())
    has_glossari = "glossari" in resp.lower() or "paraules clau" in resp.lower()
    has_negreta = "**" in resp
    print(f"{name:20s} | {words:4d} paraules | Glossari: {'SI' if has_glossari else 'NO'} | Negretes: {'SI' if has_negreta else 'NO'}")
print()
print("Compara amb els resultats de Gemma 4, Mistral, GPT a:")
print("docs/avaluacio_final_5_models.md")